In [ ]:
import pathlib
import os
from typing import List

from datasets import load_dataset
import evaluate
import torch
from transformers import (
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score


def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    pretrained_model = GPT(gptconf)
    state_dict = checkpoint['model']

    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model_dict = pretrained_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_state_dict)
    pretrained_model.load_state_dict(model_dict)
    pretrained_model.to(device)
    return pretrained_model


# ---- Configuration ----
args = {
    'task': "mnli",
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 8,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda'
}

models_and_tokenizers = [
    {"model_type": "ipa", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-merges.txt")},
    {"model_type": "normal", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-merges.txt")},
    {"model_type": "prebuilt", "model_path": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_50k/ckpt.pt",
     "tokenizer_paths": None}
]

main_output_dir = pathlib.Path("./training_outputs_mnli")
os.makedirs(main_output_dir, exist_ok=True)

for config in models_and_tokenizers:
    model_type = config["model_type"]
    model_path = config["model_path"]
    tokenizer_paths = config["tokenizer_paths"]
    output_dir = main_output_dir / f"output_{model_type}"
    os.makedirs(output_dir, exist_ok=True)

    if model_type in ["ipa", "normal"]:
        vocab_path, merges_path = tokenizer_paths
        tokenizer = load_tokenizer(vocab_path, merges_path)
    else:
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token

    base_model = load_pretrained_model(model_path, args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    # ---- Load MNLI splits ----
    train_dataset = load_dataset("iggy12345/glue-mnli-ipa", split="train", cache_dir=str(args['hf_cache_dir']))
    val_mismatched = load_dataset("iggy12345/glue-mnli-ipa", split="validation_mismatched", cache_dir=str(args['hf_cache_dir']))
    val_matched = load_dataset("iggy12345/glue-mnli-ipa", split="validation_matched", cache_dir=str(args['hf_cache_dir']))


    def flatten_multi_features(examples, features: List[str]) -> List[str]:
        separator = f'\n\n{eod_token}\n\n'
        return [separator.join(example) for example in zip(*[examples[f] for f in features])]

    def preprocess_function(examples):
        feature = flatten_multi_features(examples, ['premise', 'hypothesis'])
        return tokenizer(feature, truncation=True, max_length=args['context_size'])

    train_encoded = train_dataset.map(preprocess_function, batched=True)
    val_mismatched_encoded = val_mismatched.map(preprocess_function, batched=True)
    val_matched_encoded = val_matched.map(preprocess_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    metric = evaluate.load("glue", args['task'])

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.from_numpy(logits).argmax(dim=-1)
        hf_metrics = metric.compute(predictions=predictions, references=labels)
        hf_metrics["precision"] = precision_score(labels, predictions, average="macro")
        hf_metrics["recall"] = recall_score(labels, predictions, average="macro")
        hf_metrics["f1-score"] = f1_score(labels, predictions, average="macro")
        return hf_metrics

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        evaluation_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=1,
        metric_for_best_model="f1-score",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=100,
        logging_dir='./logs',
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_encoded,
        eval_dataset=val_matched_encoded,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train(resume_from_checkpoint=False)

    print(f"\n🎯 Matched Results for {model_type}")
    print(trainer.evaluate(eval_dataset=val_matched_encoded))

    print(f"\n🌍 Mismatched Results for {model_type}")
    print(trainer.evaluate(eval_dataset=val_mismatched_encoded))


number of parameters: 123.35M


Map: 100%|██████████| 9815/9815 [00:00<00:00, 19887.42 examples/s]
/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/slurmtmp.1421212/ipykernel_4156240/102687816.py:135: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: orugantikoundinya7 (orugantikoundinya7-ohio-st

Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
100,1.193400,1.208496,0.361488,0.360115,0.352087,0.329162
200,1.201900,1.182763,0.357922,0.355428,0.352015,0.343867
300,1.181600,1.166854,0.360774,0.360091,0.360153,0.359855
400,1.157900,1.158343,0.363627,0.361876,0.360860,0.359627
500,1.144300,1.153642,0.371472,0.372240,0.371289,0.366996
600,1.158700,1.141550,0.380031,0.378249,0.377150,0.374369
700,1.140600,1.128688,0.389404,0.388085,0.386771,0.385727
800,1.121200,1.120583,0.401732,0.401064,0.398185,0.391299
900,1.115700,1.109795,0.402038,0.405438,0.405438,0.398527
1000,1.080800,1.095457,0.414875,0.418737,0.416915,0.406765
